 Event log exploration

What a dataset holds, before and after preprocessing. Everything is driven by the dataset's
config and its fitted codec, so nothing here can disagree with what the model reads.

In [ ]:
import sys
import textwrap
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pm4py
from matplotlib.colors import PowerNorm

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config').is_dir())
sys.path.insert(0, str(ROOT))

from src import paths
from src.configs import load_dataset_config
from src.datasets.codec import DatasetCodec
from src.logs.io import read_log, read_original_log
from src.logs.keys import (
    ACTIVITY_KEY,
    CASE_ELAPSED_KEY,
    CASE_KEY,
    DAY_COS_KEY,
    DAY_SIN_KEY,
    EVENT_DELTA_KEY,
    MIN_PREFIX_KEY,
    MISSING_FEATURE,
    REMAINING_TIME_KEY,
    RESOURCE_KEY,
    SECONDS_COS_KEY,
    SECONDS_SIN_KEY,
    TIMESTAMP_KEY,
    Split,
)

plt.rcParams['figure.figsize'] = (11, 3.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

Pick the dataset here: every cell below reads it from the config and the codec.

In [ ]:
DATASET = 'sepsis'  # any dataset that has been preprocessed
CONFIG = ROOT / 'config' / 'datasets' / f'{DATASET}.yaml'

data_config = load_dataset_config(CONFIG).data
codec = DatasetCodec.load(data_config)

print(
    f'{DATASET}: {len(codec.categorical_features)} categorical and '
    f'{len(codec.numeric_features)} numeric event features, '
    f'{len(codec.activity.vocab)} activities, {len(codec.resource.vocab)} resources, '
    f'max_trace_length={codec.max_trace_length}'
)

# Raw log

The only cell that reads `original.csv`. These are the numbers that describe the dataset as
published; everything below works on the preprocessed splits instead.

In [ ]:
raw = read_original_log(data_config)
raw_cases = raw.groupby(CASE_KEY, sort=False)
raw_len = raw_cases.size()
raw_traces = raw_cases[ACTIVITY_KEY].apply(tuple)
raw_duration = (
    raw_cases[TIMESTAMP_KEY].max() - raw_cases[TIMESTAMP_KEY].min()
).dt.total_seconds() / 86400

pd.Series(
    {
        'events': len(raw),
        'cases': len(raw_cases),
        'columns': raw.shape[1],
        'activities': raw[ACTIVITY_KEY].nunique(),
        'resources': raw[RESOURCE_KEY].nunique(),
        'variants': raw_traces.nunique(),
        'variants / cases': round(raw_traces.nunique() / len(raw_traces), 3),
        'trace length mean': round(raw_len.mean(), 2),
        'trace length sd': round(raw_len.std(), 2),
        'trace length median': raw_len.median(),
        'trace length min': raw_len.min(),
        'trace length max': raw_len.max(),
        'case duration mean (days)': round(raw_duration.mean(), 2),
        'case duration sd (days)': round(raw_duration.std(), 2),
        'first event': raw[TIMESTAMP_KEY].min(),
        'last event': raw[TIMESTAMP_KEY].max(),
    }
).to_frame('value')

# Preprocessed splits

The three splits concatenated back into one log, which is what the model is trained and
evaluated on: derived columns added, cases too long to fit the sequence tensors dropped.

In [ ]:
splits = {
    split: read_log(paths.PROCESSED_SPLIT.path(dataset=DATASET, split=split), dtype={CASE_KEY: str})
    for split in Split
}
log = pd.concat(
    [frame.assign(split=str(split)) for split, frame in splits.items()], ignore_index=True
).sort_values([CASE_KEY, TIMESTAMP_KEY], kind='stable')

cases = log.groupby(CASE_KEY, sort=False)
trace_len = cases.size()
traces = cases[ACTIVITY_KEY].apply(tuple)

print(f'preprocessing kept {len(cases)}/{len(raw_cases)} cases and {len(log)}/{len(raw)} events')
log.head()

The split is out of time: test holds every case still running at the separation, so a crossing
case can start well before train ends. What keeps that leak-proof is `min_prefix_len`, which is
narrowed to the first cut point after the separation for exactly those cases - `bounded cases`
below counts them.

In [ ]:
pd.DataFrame(
    {
        str(split): {
            'events': len(frame),
            'cases': frame[CASE_KEY].nunique(),
            'activities': frame[ACTIVITY_KEY].nunique(),
            'resources': frame[RESOURCE_KEY].nunique(),
            'variants': frame.groupby(CASE_KEY, sort=False)[ACTIVITY_KEY].apply(tuple).nunique(),
            'bounded cases': (frame.groupby(CASE_KEY)[MIN_PREFIX_KEY].first() > 1).sum(),
            'first case start': frame.groupby(CASE_KEY)[TIMESTAMP_KEY].min().min(),
            'last case start': frame.groupby(CASE_KEY)[TIMESTAMP_KEY].min().max(),
        }
        for split, frame in splits.items()
    }
)

In [ ]:
starts = cases[TIMESTAMP_KEY].min()
split_of_case = cases['split'].first()

fig, ax = plt.subplots()
for split in splits:
    ax.hist(starts[split_of_case == str(split)], bins=60, label=str(split), alpha=0.8)
ax.set(xlabel='case start', ylabel='cases', title='out-of-time split')
ax.legend()
plt.tight_layout()

# Trace length

The cutoff is read off the raw distribution at `data.max_seq_len_percentile`, so the raw log is
what shows where it bites.

In [ ]:
print('Preprocessed trace length quantiles (events):')
print(trace_len.quantile([0.5, 0.75, 0.9, 0.95, 0.99, 1.0]).to_string())
print(
    f'\nraw cases above max_trace_length={codec.max_trace_length}: '
    f'{(raw_len > codec.max_trace_length).mean():.2%}'
)
print(
    f'Raw events kept under that cutoff: '
    f'{raw_len.clip(upper=codec.max_trace_length).sum() / raw_len.sum():.2%}'
)

fig, axes = plt.subplots(1, 2)
axes[0].hist(raw_len, bins=min(60, int(raw_len.max())))
axes[0].axvline(codec.max_trace_length, color='red', ls='--', label='max_trace_length')
axes[0].set(xlabel='raw trace length', ylabel='cases')
axes[0].legend()
axes[1].plot(np.sort(raw_len), np.linspace(0, 1, len(raw_len)))
axes[1].axvline(codec.max_trace_length, color='red', ls='--')
axes[1].set(xlabel='raw trace length', ylabel='cumulative share of cases', xscale='log')
plt.tight_layout()

# Activities

Frequencies come in two flavours, and an activity that repeats within a case scores very
differently on them: `absolute` and `relative frequency` count events, `case frequency` and
`case coverage` count the cases holding at least one, and `mean repetitions` is the ratio
between the two. `as start` and `as end` count the cases an activity opens and closes.

In [ ]:
occurrences = log[ACTIVITY_KEY].value_counts()
containing_cases = log.groupby(ACTIVITY_KEY)[CASE_KEY].nunique()

activity_stats = (
    pd.DataFrame(
        {
            'absolute frequency': occurrences,
            'relative frequency': (occurrences / len(log)).round(4),
            'case frequency': containing_cases,
            'case coverage': (containing_cases / log[CASE_KEY].nunique()).round(4),
            'mean repetitions': (occurrences / containing_cases).round(2),
            'as start': cases[ACTIVITY_KEY].first().value_counts(),
            'as end': cases[ACTIVITY_KEY].last().value_counts(),
        }
    )
    .fillna(0)
    .astype({'as start': int, 'as end': int})
    .sort_values('absolute frequency', ascending=False)
)
activity_stats

## Where the activities sit

Each event is placed at its relative position in its case, 0.0 for the first and 1.0 for the
last, so cases of different lengths can be read on one axis. The two panels normalize that
same grid in opposite directions, and answer opposite questions: the heatmap normalizes each
row, so it says *where an activity happens*, with the rows ordered by mean position; the area
chart normalizes each column, so it says *what happens at that point of a case*.

In [ ]:
POSITION_BINS = 10
TOP_ACTIVITIES = 15

# 0.0 for the first event of a case, 1.0 for the last; single-event cases sit at 0.0.
position = cases.cumcount() / (cases[ACTIVITY_KEY].transform('size') - 1).replace(0, 1)
binned = np.clip((position * POSITION_BINS).astype(int), 0, POSITION_BINS - 1)
centers = (np.arange(POSITION_BINS) + 0.5) / POSITION_BINS

top = activity_stats.head(TOP_ACTIVITIES).index
ordered = position.groupby(log[ACTIVITY_KEY]).mean().loc[top].sort_values().index

where = pd.crosstab(log[ACTIVITY_KEY], binned, normalize='index').loc[ordered]
mix = pd.crosstab(binned, log[ACTIVITY_KEY], normalize='index').reindex(columns=ordered).fillna(0.0)
mix.index = centers

fig, axes = plt.subplots(1, 2, figsize=(13, max(4.0, 0.32 * len(top))))
# An activity spread evenly over a case sits at 1/POSITION_BINS in every cell, two orders below
# the ones that only ever open or close a case, so a linear ramp would leave the middle blank.
image = axes[0].imshow(where, aspect='auto', cmap='Blues', norm=PowerNorm(gamma=0.4, vmax=1.0))
axes[0].set_yticks(range(len(where.index)), where.index)
axes[0].set_xticks(range(POSITION_BINS), [f'{center:.0%}' for center in centers], rotation=90)
axes[0].set(xlabel='relative position in case', title='where an activity happens')
axes[0].grid(False)
fig.colorbar(image, ax=axes[0], shrink=0.8)

mix.plot.area(ax=axes[1], cmap='tab20', legend=False)
axes[1].set(
    xlabel='relative position in case',
    ylabel='share of events',
    xlim=(centers[0], centers[-1]),
    ylim=(0.0, 1.0),
    title='what happens at that point',
)
axes[1].legend(bbox_to_anchor=(1.01, 1.0), loc='upper left', fontsize=7)
plt.tight_layout()

## Directly-follows graph

An edge `a -> b` counts every case where `b` immediately follows `a`; the diagram keeps only the
`MAX_DFG_EDGES` strongest edges, since a full graph over every activity pair is unreadable. Start
and end arrows mark the activities a case opens and closes with, matching `as start`/`as end`
above.

In [ ]:
from pm4py.visualization.dfg import visualizer as dfg_visualizer

MAX_DFG_EDGES = 25

dfg, dfg_starts, dfg_ends = pm4py.discover_dfg(
    log, activity_key=ACTIVITY_KEY, timestamp_key=TIMESTAMP_KEY, case_id_key=CASE_KEY
)
# Trimmed here rather than left to the visualizer's own max-edges cutoff: it can drop a start/end
# activity whose only edges fall below the cutoff, but still list it, and then crash.
top_edges = dict(sorted(dfg.items(), key=lambda item: item[1], reverse=True)[:MAX_DFG_EDGES])
kept_activities = {activity for edge in top_edges for activity in edge}
top_starts = {
    activity: count for activity, count in dfg_starts.items() if activity in kept_activities
}
top_ends = {activity: count for activity, count in dfg_ends.items() if activity in kept_activities}

dfg_parameters = dfg_visualizer.Variants.FREQUENCY.value.Parameters
dfg_visualizer.apply(
    top_edges,
    variant=dfg_visualizer.Variants.FREQUENCY,
    parameters={
        dfg_parameters.START_ACTIVITIES: top_starts,
        dfg_parameters.END_ACTIVITIES: top_ends,
        dfg_parameters.FORMAT: 'svg',
        dfg_parameters.RANKDIR: 'LR',
    },
)

# Variants

A variant is one distinct activity sequence, however many cases walk it.

In [ ]:
variants = traces.value_counts()
coverage = variants.cumsum() / variants.sum()

print(f'Unique traces: {len(variants)} over {len(traces)} cases')
print(
    f'Singleton variants: {(variants == 1).sum()} ({(variants == 1).mean():.2%} of variants, '
    f'{(variants == 1).sum() / len(traces):.2%} of cases)\n'
)
for k in (1, 5, 10, 25, 50, 100):
    if k <= len(variants):
        print(f'top {k:>4} variants cover {coverage.iloc[k - 1]:.2%} of cases')

One row per variant, one cell per position, coloured by activity: the block of shared colour
down the left is the prefix the frequent variants agree on, and where the rows stop agreeing is
where a prefix stops determining its suffix. Trailing grey is a variant that has already ended.

The same variants in full, since a long one does not survive a table cell.

In [ ]:
TOP_VARIANTS = 15
top_variants = variants.head(TOP_VARIANTS)

for rank, (variant, count) in enumerate(top_variants.items(), start=1):
    print(f'#{rank:<3} {count:>6} cases  {count / len(traces):>6.2%}  {len(variant):>3} events')
    print(
        textwrap.fill(
            ' > '.join(variant),
            width=110,
            initial_indent=' ' * 6,
            subsequent_indent=' ' * 8,
        )
    )
    print()

## Time

Seven columns come out of the event's timestamp; `ts_prev`, `ts_start` and `rtime` are in
minutes.
- `ts_prev` (event delta): minutes since the previous event in the same case, 0.0 at the first
  event. **Encoder input, only when `data.event_features` names it.**
- `ts_start` (case elapsed): minutes since the first event in the same case, 0.0 at the first
  event. **Encoder input, on the same condition.**
- `rtime` (remaining time): minutes until the case ends, 0.0 at the last event. **Decoder target
  only** - a Gaussian regression head trained against it, never something the encoders read.
- `day_sin`/`day_cos`: the weekday of the event's own timestamp, as a point on the unit circle
  rather than 0 (Monday) to 6 (Sunday) raw, so Sunday sits next to Monday instead of six days
  away. Read off the event rather than the case, so it says nothing about progress through the
  case, only the work calendar behind it. **Encoder input, only when `data.event_features`
  names it.**
- `seconds_sin`/`seconds_cos`: the second of the event's own timestamp since midnight, 0 to
  86399, on the same circular treatment as `day_sin`/`day_cos`.

The two duration proxies and all four calendar columns are always computed during
preprocessing, whether or not the current dataset's config feeds them to the model - the cell below prints which is the case for the
selected `DATASET`. The three duration columns are plotted on their raw, unstandardized scale
for readability; the model itself reads them z-scored (and optionally log1p'd, per
`data.log_scaled_features` for the two proxies and `data.log_scaled_remaining_time` for
`rtime`).

The two plots below match this split: the encoder's actual inputs first, the decoder's target
kept separate and clearly labelled as such.

In [ ]:
delta = log.loc[cases.cumcount() > 0, EVENT_DELTA_KEY]  # ts_prev, first event of each case excluded
elapsed = log.loc[cases.cumcount() > 0, CASE_ELAPSED_KEY]  # ts_start, same exclusion
remaining = log[REMAINING_TIME_KEY] / 60 / 24  # rtime, days

print('ts_prev (event delta, minutes, first event of each case excluded)')
print(delta.describe().round(2).to_string())
print(f'\nzero deltas: {(delta == 0).mean():.2%}')
print('\nts_start (case elapsed, minutes, first event of each case excluded)')
print(elapsed.describe().round(2).to_string())

# Linear scale, the model's own: no dataset log-scales a duration. Binned up
# to p99.5 rather than clipped there, so the tail thins out naturally instead of piling into a
# fake spike at the edge; the p99 line is only a reference mark, not where anything is cut.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, values, xlabel, title, color in (
    (axes[0], delta, 'ts_prev (min)', 'encoder input: ts_prev', 'tab:blue'),
    (axes[1], elapsed, 'ts_start (min)', 'encoder input: ts_start', 'tab:blue'),
    (axes[2], remaining, 'rtime (days)', 'decoder target: rtime', 'tab:orange'),
):
    p99 = values.quantile(0.99)
    window = values.quantile(0.995)
    ax.hist(values, bins=60, range=(0, window), color=color)
    ax.axvline(p99, color='red', ls='--', label='p99')
    ax.set(xlabel=xlabel, ylabel='events', title=title)
    ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
active_calendar = [
    key
    for key in (DAY_SIN_KEY, DAY_COS_KEY, SECONDS_SIN_KEY, SECONDS_COS_KEY)
    if key in data_config.event_features
]
print(
    f'{DATASET}: '
    + (
        f'{", ".join(active_calendar)} fed to the encoders'
        if active_calendar
        else 'no calendar column fed to the encoders'
    )
)

if active_calendar:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].scatter(log[DAY_COS_KEY], log[DAY_SIN_KEY], s=2, alpha=0.05)
    axes[0].set(
        xlabel='cos(day in week)',
        ylabel='sin(day in week)',
        title='encoder input: day, cyclical',
        aspect='equal',
    )
    axes[1].scatter(log[SECONDS_COS_KEY], log[SECONDS_SIN_KEY], s=2, alpha=0.05)
    axes[1].set(
        xlabel='cos(second in day)',
        ylabel='sin(second in day)',
        title='encoder input: time, cyclical',
        aspect='equal',
    )
    plt.tight_layout()

# Event attributes

Read off the fitted codec rather than off the config, so this is exactly what the model embeds.
`channel` is the rule `DatasetCodec.fit` applied: a column's pandas dtype decides it, which is
why `day_sin`, `day_cos`, `seconds_sin` and `seconds_cos` land among the numeric features and
are standardized rather than embedded. `level` is read off the data instead - the model reads
both the same way, a channel on every event, and a case-level column is simply one whose value
never changes where it is filled in at all. `vocab` counts the codec's own entries, `<MISSING>`
among them wherever a categorical column has gaps. Activities and resources are channels of
their own, covered above.

In [ ]:
categorical = [column.column for column in codec.categorical_features]
numeric = [column.column for column in codec.numeric_features]

# A gap is not a value: categorical gaps carry MISSING_FEATURE and numeric ones a NaN, and both
# have to drop out before a column can be called constant within its case.
present = log[categorical + numeric].mask(log[categorical + numeric].eq(MISSING_FEATURE))
per_case = present.groupby(log[CASE_KEY], sort=False).nunique().max()  # 1 => constant in a case

pd.DataFrame(
    [
        {
            'feature': column.column,
            'channel': 'categorical',
            'dtype': str(log[column.column].dtype),
            'level': 'case' if per_case[column.column] <= 1 else 'event',
            'vocab': len(column.vocab),
            'missing': round(log[column.column].eq(MISSING_FEATURE).mean(), 4),
            'mean': None,
            'sd': None,
        }
        for column in codec.categorical_features
    ]
    + [
        {
            'feature': column.column,
            'channel': 'numeric',
            'dtype': str(log[column.column].dtype),
            'level': 'case' if per_case[column.column] <= 1 else 'event',
            'vocab': log[column.column].nunique(),
            'missing': round(log[column.column].isna().mean(), 4),
            'mean': round(column.mean, 3),
            'sd': round(column.std, 3),
        }
        for column in codec.numeric_features
    ]
).set_index('feature')

## Numeric features

In [ ]:
log[numeric].describe().T.round(3)

In [ ]:
if numeric:
    columns = min(4, len(numeric))
    rows = -(-len(numeric) // columns)
    fig, axes = plt.subplots(rows, columns, figsize=(3 * columns, 2.4 * rows))
    # The grid can hold more cells than there are features; the leftover axes are hidden below.
    flat = np.ravel([axes])
    for ax, name in zip(flat, numeric, strict=False):
        values = log[name].dropna()
        # Full distribution binned up to p99.5, not clipped there, so the tail thins out
        # naturally instead of piling into a fake spike at the edge; p99 is a reference mark.
        p99 = values.quantile(0.99)
        window = values.quantile(0.995)
        ax.hist(values, bins=40, range=(0, window))
        ax.axvline(p99, color='red', ls='--', linewidth=1)
        ax.set_title(name, fontsize=9)
    for ax in flat[len(numeric) :]:
        ax.axis('off')
    plt.tight_layout()

## Categorical features

In [ ]:
for column in codec.categorical_features:
    values = log[column.column]
    print(
        f'{column.column}  ({len(column.vocab)} values in the codec, '
        f'{values.eq(MISSING_FEATURE).mean():.1%} missing)'
    )
    print(values.value_counts(normalize=True).round(4).head(8).to_string(), '\n')